# Averaging the same-module interaction splits

Pools the five random half-splits fitted per module by
`04_FitSameModuleInteraction_SingleSplit.ipynb` into one coefficient per gene
per module, and writes the table the Figure 5 notebooks read.

Within each gene the five splits are FDR corrected together, coefficients
failing the cutoff are set to zero, and the five are averaged. A gene therefore
ends up non-zero if its within-module interaction was significant in at least
one of the five splits.

## Setup

In [ ]:
library(data.table)

# NOTE: Main.R is deliberately not sourced. It loads ICtest, whose Ops method
# conflicts with the S7 objects in ggplot2 4.x.

SPLIT_DIR  <- "/home/eraslab1/Projects/E3Ligase/analysisSingle/Notebooks/CombinatorialPerturbations/RDSFiles"
OUT_FILE   <- "outputs/ComboEffects_doublesResampleRes.rds"

MODULES    <- 0:4        # module 5 has too few same-module doubles to split
ITERATIONS <- 1:5
FDR_CUTOFF <- 0.1

## One column per module

The original handled module 0 in three cells and modules 1 to 4 in an identical
loop, purely so the first could create the frame the others merged into. Both
paths did the same thing, so there is one function here.

In [ ]:
averageSplits <- function(guideGroup) {
    splits <- do.call(rbind, lapply(ITERATIONS, function(iteration) {
        k <- readRDS(file.path(SPLIT_DIR, sprintf("ComboEffects_doublesResample_KO_%d_%d.rds",
                                                  guideGroup, iteration)))
        k[grep(":", k$term), ]        # keep the interaction term, drop the two main effects
    }))

    splits <- data.table(splits)
    splits[, FDR := p.adjust(p.value, method = "fdr", n = length(p.value)), by = respGene]
    splits$estimateAdj <- splits$estimate
    splits[splits$FDR > FDR_CUTOFF, "estimateAdj"] <- 0
    splits[, meanEstimateAdj := mean(estimateAdj), by = respGene]
    splits <- data.frame(splits)

    splits <- unique(splits[, c("meanEstimateAdj", "respGene")])
    colnames(splits) <- c(sprintf("K%d:K%d", guideGroup, guideGroup), "respGene")
    splits
}

allDF <- averageSplits(MODULES[1])
for (guideGroup in MODULES[-1]) {
    allDF <- merge(allDF, averageSplits(guideGroup), by = "respGene")
}

dim(allDF)

## Write it

In [ ]:
saveRDS(allDF, OUT_FILE)
head(allDF)